<a href="https://colab.research.google.com/github/shay-bagaria/resistancemap-za/blob/main/notebooks/ResistanceMap_ZA_03_Mutation_Identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Data Loading
# ==============================================================================
import os
import json
import requests
import pandas as pd
from Bio import SeqIO

print("Initialising Mutation Identification Engine...")

# Ensure output directory exists for our results
os.makedirs("data/mutation_tables", exist_ok=True)

# Define the path to the KZN FASTA file from Notebook 2
fasta_path = "data/raw_sequences/kzn_hiv_pol.fasta"
sequences = []

try:
    # Load the sequences into memory
    for record in SeqIO.parse(fasta_path, "fasta"):
        sequences.append({"header": record.id, "sequence": str(record.seq)})

    print(f"✅ Successfully loaded {len(sequences)} KZN sequences for analysis.")
except FileNotFoundError:
    print(f"❌ Error: Could not find {fasta_path}. Ensure Notebook 2 ran successfully in this session.")

In [ ]:
# ==============================================================================
# CELL 2: Stanford Sierra API Interfacing
# ==============================================================================

# Stanford's official GraphQL endpoint for sequence analysis
SIERRA_URL = "https://hivdb.stanford.edu/graphql"

# We construct a GraphQL query that asks Stanford for:
# 1. The mutations found in the Reverse Transcriptase (RT) and Protease (PR) regions
# 2. The drug resistance scores for specific ARVs
graphql_query = """
    query($sequences: [String]!) {
      currentVersion { text }
      sequenceAnalysis(sequences: $sequences) {
        inputSequence { header }
        mutations {
          primaryType text
        }
        drugResistance {
          drug { name displayAbbr }
          score
          level
        }
      }
    }
"""

print("Transmitting sequences to Stanford University HIVDB...")

# Extract just the raw sequence strings for the payload (testing first 10 sequences)
sequence_payload = [seq["sequence"] for seq in sequences[:10]]

try:
    response = requests.post(
        SIERRA_URL,
        json={
            "query": graphql_query,
            "variables": {"sequences": sequence_payload}
        }
    )

    if response.status_code == 200:
        stanford_results = response.json()
        print("✅ Success: Stanford API processed the sequences and returned clinical data.")
    else:
        print(f"❌ API Error: HTTP {response.status_code}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
# ==============================================================================
# CELL 3: Data Parsing & CSV Export
# ==============================================================================

print("Extracting mutation profiles and constructing resistance matrix...")

mutation_tally = []

# Parse the JSON response from Stanford
analyses = stanford_results.get("data", {}).get("sequenceAnalysis", [])

for analysis in analyses:
    # Extract mutations
    for mut in analysis.get("mutations", []):
        mut_text = mut.get("text")

        # We only want to log severe, drug-associated mutations, not natural variants
        if mut.get("primaryType") == "Major":
            mutation_tally.append(mut_text)

# Convert the list into a structured Pandas DataFrame to count frequencies
if mutation_tally:
    df_mutations = pd.DataFrame(mutation_tally, columns=["Mutation"])
    frequency_table = df_mutations.value_counts().reset_index(name="Count")
    frequency_table = frequency_table.rename(columns={0: "Mutation"})

    # Calculate percentage prevalence across the tested batch
    batch_size = len(analyses)
    frequency_table["Prevalence (%)"] = (frequency_table["Count"] / batch_size) * 100

    # Export to a clean CSV file
    output_csv = "data/mutation_tables/kzn_mutation_frequencies.csv"
    frequency_table.to_csv(output_csv, index=False)

    print("\n✅ Resistance Matrix Compiled.")
    print("Top 5 Major Resistance Mutations found in this batch:")
    print(frequency_table.head())
    print(f"\n📁 Data exported securely to {output_csv}")
else:
    print("\n⚠️ No major resistance mutations detected in this specific sample batch.")